# Week 11A — Deep Semantic and Instance Segmentation

**MCTA 4364 Machine Vision** · Session 1 of Week 11 · 80 minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W11_segmentation_anomaly/W11A_deep_segmentation.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W11_segmentation_anomaly/W11A_deep_segmentation.ipynb)

Boxes are coarse. Measuring the area of a defect, picking a part by its outline, guiding a sealant robot along an edge or computing the
fill level of a container needs **pixel-accurate** masks. In Week 4 we segmented with thresholds, edges and watershed; today a CNN learns to label
**every pixel**. You will train a **U-Net** from scratch on the conveyor scenes, see why **skip connections** matter, choose losses and metrics that
survive class imbalance, and then use a pretrained **instance segmentation** model (YOLO-seg / Mask R-CNN) on real images.

### Learning outcomes
By the end of this session you will be able to:
1. Distinguish **semantic**, **instance** and **panoptic** segmentation and pick the right one for a task.
2. Explain how encoder–decoder networks recover resolution (**transposed convolution**, **skip connections**, **dilated convolution**).
3. Evaluate masks with **IoU / mIoU / Dice**, and explain why pixel accuracy misleads.
4. Train a **U-Net** with a cross-entropy + **Dice** loss and interpret per-class results.
5. Explain how **Mask R-CNN** and **YOLO-seg** produce one mask per object, and export masks as polygon labels.

### Session plan (80 min)
| Time | Section | Activity |
|---|---|---|
| 0–8 | 1 | Segmentation tasks |
| 8–22 | 2–3 | From classifiers to dense prediction; metrics and imbalance |
| 22–50 | 4 | U-Net: training runs in the background while we discuss skips and losses |
| 50–68 | 5 | Instance segmentation: Mask R-CNN, YOLO-seg; masks to polygons |
| 68–80 | 8 | Quiz and exit ticket |
| Home | 6–7 | Exercises with self-checks, challenge |

**Hardware:** about 4 minutes on a laptop CPU (three small U-Nets), under a minute on a GPU. YOLO11n-seg weights (6 MB) download once.

## 0. Setup

In [ ]:
# Setup: works in Google Colab, Kaggle, Jupyter/VS Code on your laptop, and CI.
import importlib.util
import os
import subprocess
import sys

REPO_URL = "https://github.com/hasanzaki/MCTA-4364-Machine-Vision.git"
if not os.path.exists("resources/scripts/cvhelpers.py"):
    if os.path.exists("../../resources/scripts/cvhelpers.py"):  # opened from its own folder
        os.chdir("../..")
    else:  # Colab / Kaggle: fetch the course helpers
        if not os.path.isdir("MCTA-4364-Machine-Vision"):
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
        os.chdir("MCTA-4364-Machine-Vision")
print("Working directory:", os.getcwd())

# Install only what is missing. PyTorch is deliberately NOT pip-installed here:
# that would replace your laptop's CUDA build with a CPU-only one (see SETUP.md).
needed = {"cv2": "opencv-python", "skimage": "scikit-image", "ipywidgets": "ipywidgets",
          "matplotlib": "matplotlib", "scipy": "scipy", "plotly": "plotly",
          "ultralytics": "ultralytics"}
missing = [pkg for mod, pkg in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
if importlib.util.find_spec("torch") is None:
    raise ImportError("PyTorch is missing. Install it with CUDA support as described in "
                      "SETUP.md, or use Google Colab where it is preinstalled.")
sys.path.append("resources/scripts")

In [ ]:
import time

import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from torch import nn

import partsim
from cvhelpers import SMOKE, check, data_dir, interact, load_sample, opencv_sample, pipeline_diagram, quiz, show
from dlhelpers import count_params, hw_profile, load_pretrained, set_seed

plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 11})
HW = hw_profile()
DEVICE = HW["device"]
set_seed(0)
NAMES = ["belt"] + partsim.SCENE_CLASSES
COLOURS = np.array([[40, 40, 40], [234, 67, 53], [52, 168, 83], [66, 133, 244], [251, 188, 5]], np.uint8)   # RGB per class
N_TRAIN, N_TEST, EPOCHS = (16, 8, 1) if SMOKE else (200, 30, 12)
SIZE = (160, 208)                                                        # (height, width): scenes downscaled by 3


def make_split(n, seed):
    """Greyscale scenes and their per-pixel class labels (0 = belt, 1..4 = part classes)."""
    rng = np.random.default_rng(seed)
    X, Y, raw = [], [], []
    for _ in range(n):
        img, b, l, masks = partsim.make_scene(rng.integers(1 << 31), return_masks=True)
        lab = np.zeros(img.shape[:2], np.uint8)
        for m, c in zip(masks, l):
            lab[m] = c + 1
        X.append(cv2.resize(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), SIZE[::-1], interpolation=cv2.INTER_AREA))
        Y.append(cv2.resize(lab, SIZE[::-1], interpolation=cv2.INTER_NEAREST)); raw.append((img, b, l, masks))
    return torch.tensor(np.stack(X))[:, None].float() / 255, torch.tensor(np.stack(Y)).long(), raw


t0 = time.perf_counter()
X_train, Y_train, _ = make_split(N_TRAIN, 1)
X_test, Y_test, raw_test = make_split(N_TEST, 2)
freq = np.bincount(Y_train.numpy().ravel(), minlength=5) / Y_train.numel()
print(f"{N_TRAIN} training / {N_TEST} test scenes at {SIZE[1]} x {SIZE[0]} ({time.perf_counter() - t0:.0f} s)")
print("pixel share per class:", {n: f"{f:.2%}" for n, f in zip(NAMES, freq)})

## 1. Three segmentation tasks

| task | output | question it answers | typical models |
|---|---|---|---|
| **semantic** | one class per pixel | *which pixels are bolt?* | FCN, U-Net, DeepLab, SegFormer |
| **instance** | one mask per object (+ class, score) | *how many bolts, and which pixels belong to each?* | Mask R-CNN, YOLO-seg, SOLO |
| **panoptic** | class per pixel **and** instance IDs for countable things | *everything in the scene* | Panoptic FPN, Mask2Former, OneFormer |

Background regions without countable instances (belt, sky, road) are called **stuff**; countable objects are **things**.

In [ ]:
img, b, l, masks = raw_test[0]
sem = np.zeros(img.shape[:2], np.uint8)
for m, c in zip(masks, l):
    sem[m] = c + 1
rng = np.random.default_rng(3)
inst_vis = np.full(img.shape, 40, np.uint8)
for m in masks:
    inst_vis[m] = rng.integers(80, 255, 3)
pan_vis = COLOURS[sem][..., ::-1].copy()
for m in masks:
    cs, _ = cv2.findContours(m.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    cv2.drawContours(pan_vis, cs, -1, (255, 255, 255), 2)
fig, axes = plt.subplots(1, 4, figsize=(22, 4.4))
axes[0].imshow(img[..., ::-1]); axes[0].set_title("image")
axes[1].imshow(COLOURS[sem]); axes[1].set_title("semantic: class per pixel (touching parts merge)")
axes[2].imshow(inst_vis[..., ::-1]); axes[2].set_title(f"instance: {len(masks)} separate masks")
axes[3].imshow(pan_vis[..., ::-1]); axes[3].set_title("panoptic: classes + instance outlines")
for ax in axes:
    ax.axis("off")
plt.suptitle("Figure 1.1 - The same scene as semantic, instance and panoptic segmentation", fontweight="bold")
plt.tight_layout(); plt.show()

## 2. From a classifier to dense prediction

A classification CNN shrinks the image by 32× to get **context** (a large receptive field). Segmentation needs both context *and* the original
resolution. Three tools:

- **Up-sampling** in the decoder: bilinear interpolation or a **transposed convolution** (a learnable up-sampling, sometimes called "deconvolution").
- **Skip connections** (U-Net, Ronneberger et al., 2015): concatenate encoder features of the same resolution into the decoder, so fine detail
  (edges, small objects) lost in pooling comes back.
- **Dilated (atrous) convolutions** (DeepLab): spread the kernel taps apart to enlarge the receptive field **without** down-sampling.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(21, 4.6))
# transposed convolution in 1D: each input value "stamps" a scaled copy of the kernel
x_in = np.array([1.0, 2.0, 0.5]); k = np.array([1.0, 0.5])
y = np.zeros(2 * len(x_in))
for i, v in enumerate(x_in):
    y[2 * i:2 * i + 2] += v * k
axes[0].stem(np.arange(len(x_in)) * 2 + 0.5, x_in, linefmt="C0-", markerfmt="C0o", basefmt=" ", label="input (3 values)")
axes[0].stem(np.arange(len(y)), y, linefmt="C3-", markerfmt="C3s", basefmt=" ", label="transposed conv output (6 values)")
axes[0].set_title("transposed convolution, stride 2: learnable up-sampling"); axes[0].legend(); axes[0].grid(alpha=0.3)
# dilation
for ax_i, (d, col) in enumerate([(1, "#1a73e8"), (2, "#188038"), (4, "#d93025")]):
    pts = np.array([(i, j) for i in (-d, 0, d) for j in (-d, 0, d)])
    axes[1].scatter(pts[:, 0] + ax_i * 12, pts[:, 1], s=120, color=col, label=f"dilation {d}: covers {2 * d + 1} x {2 * d + 1}")
axes[1].set_aspect("equal"); axes[1].set_xticks([]); axes[1].set_yticks([]); axes[1].legend(loc="lower center", fontsize=8)
axes[1].set_title("dilated 3 x 3 kernels: 9 weights, growing receptive field")
axes[1].set_ylim(-9, 6)
pipeline_diagram(["input\n1 x H x W", "encoder\n(conv + pool) x 3", "bottleneck\nH/8 x W/8", "decoder\n(up-conv + skip) x 3",
                  "1 x 1 conv\nC class scores"], colors=["#f1f3f4", "#e8f0fe", "#fef7e0", "#e6f4ea", "#fce8e6"], ax=axes[2], fontsize=9)
axes[2].set_title("U-Net: skips join encoder and decoder at each scale")
plt.suptitle("Figure 2.1 - Recovering resolution: transposed convolution, dilation and the U-Net layout", fontweight="bold")
plt.tight_layout(); plt.show()

## 3. Metrics and class imbalance

On the conveyor, **96% of pixels are belt**. A "model" that predicts *belt everywhere* has 96% pixel accuracy and is useless. Use per-class overlap:

$$\mathrm{IoU}_c = \frac{|P_c \cap G_c|}{|P_c \cup G_c|}, \qquad \mathrm{mIoU} = \frac{1}{C}\sum_c \mathrm{IoU}_c, \qquad
\mathrm{Dice}_c = \frac{2|P_c \cap G_c|}{|P_c| + |G_c|} = \frac{2\,\mathrm{IoU}_c}{1 + \mathrm{IoU}_c}.$$

The same imbalance hurts training: the cross-entropy loss is dominated by the easy belt pixels. The **Dice loss** $1 - \mathrm{Dice}$ (computed on
soft probabilities) weighs every class equally, whatever its size; combining both is a robust default.

In [ ]:
def per_class_iou(pred, gt, n=5):
    pred, gt = np.asarray(pred), np.asarray(gt)
    return np.array([((pred == c) & (gt == c)).sum() / max(((pred == c) | (gt == c)).sum(), 1) for c in range(n)])


all_belt = np.zeros_like(Y_test.numpy())
iou_b = per_class_iou(all_belt, Y_test.numpy())
print(f"'belt everywhere': pixel accuracy {(all_belt == Y_test.numpy()).mean():.1%} | mIoU {iou_b.mean():.3f} | per class {np.round(iou_b, 2)}")

## 4. Training a U-Net

A compact U-Net (four scales, 12–96 channels, ~0.3 M parameters). We train three variants for the same number of steps:
1. **U-Net, cross-entropy + Dice** (the recommended recipe);
2. **no skip connections** (a plain encoder–decoder);
3. **U-Net, cross-entropy only**.

In [ ]:
def conv_block(i, o):
    return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(inplace=True),
                         nn.Conv2d(o, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(inplace=True))


class UNet(nn.Module):
    def __init__(self, n_classes=5, ch=(12, 24, 48, 96), skips=True):
        super().__init__()
        self.skips = skips
        self.enc = nn.ModuleList([conv_block(1 if i == 0 else ch[i - 1], ch[i]) for i in range(len(ch))])
        self.up = nn.ModuleList([nn.ConvTranspose2d(ch[i], ch[i - 1], 2, stride=2) for i in range(len(ch) - 1, 0, -1)])
        self.dec = nn.ModuleList([conv_block(ch[i - 1] * (2 if skips else 1), ch[i - 1]) for i in range(len(ch) - 1, 0, -1)])
        self.head = nn.Conv2d(ch[0], n_classes, 1)

    def forward(self, x):
        feats = []
        for i, block in enumerate(self.enc):
            x = block(x if i == 0 else F.max_pool2d(x, 2))
            feats.append(x)
        for k, (up, block) in enumerate(zip(self.up, self.dec)):
            x = up(x)
            x = block(torch.cat([x, feats[-2 - k]], 1) if self.skips else x)
        return self.head(x)


def soft_dice_loss_ref(logits, target, n=5, eps=1.0):
    p = logits.softmax(1)
    oh = F.one_hot(target, n).permute(0, 3, 1, 2).float()
    dice = (2 * (p * oh).sum((0, 2, 3)) + eps) / ((p + oh).sum((0, 2, 3)) + eps)
    return 1 - dice.mean()


def train_unet(skips=True, use_dice=True, epochs=EPOCHS, batch=8):
    set_seed(0)
    net = UNet(skips=skips).to(DEVICE)
    opt = torch.optim.AdamW(net.parameters(), 3e-3)
    steps = epochs * ((len(X_train) + batch - 1) // batch)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, 5e-3, total_steps=steps)
    losses = []
    for ep in range(epochs):
        net.train()
        for i in torch.randperm(len(X_train)).split(batch):
            x, y = X_train[i].to(DEVICE), Y_train[i].to(DEVICE)
            out = net(x)
            loss = F.cross_entropy(out, y) + (soft_dice_loss_ref(out, y) if use_dice else 0)
            opt.zero_grad(); loss.backward(); opt.step(); sched.step()
            losses.append(loss.item())
    net.eval()
    with torch.no_grad():
        pred = torch.cat([net(X_test[i:i + 16].to(DEVICE)).argmax(1).cpu() for i in range(0, len(X_test), 16)])
    return net, pred.numpy(), losses


runs = {}
for name, kw in [("U-Net, CE + Dice", dict()), ("no skip connections", dict(skips=False)), ("U-Net, CE only", dict(use_dice=False))]:
    t0 = time.perf_counter()
    net, pred, losses = train_unet(**kw)
    runs[name] = dict(net=net, pred=pred, losses=losses, iou=per_class_iou(pred, Y_test.numpy()), time=time.perf_counter() - t0)
    print(f"{name:<22} mIoU {runs[name]['iou'].mean():.3f} | per class {np.round(runs[name]['iou'], 2)} | {runs[name]['time']:.0f} s")
print(f"model size: {count_params(runs['U-Net, CE + Dice']['net']):,} parameters")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 4.3), gridspec_kw=dict(width_ratios=[1.3, 1]))
xs = np.arange(5)
for k, (name, r) in enumerate(runs.items()):
    axes[0].bar(xs + (k - 1) * 0.27, r["iou"], 0.27, label=f"{name} (mIoU {r['iou'].mean():.2f})")
axes[0].set_xticks(xs); axes[0].set_xticklabels(NAMES); axes[0].set_ylim(0, 1.05); axes[0].set_ylabel("IoU"); axes[0].legend(fontsize=8, loc="lower right")
axes[0].grid(alpha=0.3, axis="y"); axes[0].set_title("per-class IoU on the test scenes")
for name, r in runs.items():
    axes[1].plot(np.convolve(r["losses"], np.ones(10) / 10, mode="valid"), label=name)
axes[1].set_xlabel("step"); axes[1].set_ylabel("training loss (smoothed)"); axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
axes[1].set_title("training curves (CE + Dice and CE-only losses have different scales)")
plt.suptitle("Figure 4.1 - Skip connections and the Dice loss matter most for the small classes", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
def overlay(gray_img, labels, alpha=0.55):
    base = np.repeat((gray_img * 255).astype(np.uint8)[..., None], 3, 2)
    col = COLOURS[labels]
    out = base.copy(); m = labels > 0
    out[m] = ((1 - alpha) * base[m] + alpha * col[m]).astype(np.uint8)
    return out


def seg_demo(scene=0, model="U-Net, CE + Dice"):
    scene = min(scene, len(X_test) - 1)
    g = X_test[scene, 0].numpy(); p = runs[model]["pred"][scene]; y = Y_test[scene].numpy()
    err = (p != y)
    fig, axes = plt.subplots(1, 4, figsize=(21, 4))
    axes[0].imshow(g, cmap="gray"); axes[0].set_title(f"test scene {scene}")
    axes[1].imshow(overlay(g, y)); axes[1].set_title("ground truth")
    axes[2].imshow(overlay(g, p)); axes[2].set_title(f"{model}: IoU per class {np.round(per_class_iou(p, y), 2)}", fontsize=9)
    axes[3].imshow(err, cmap="Reds"); axes[3].set_title(f"errors: {err.mean():.2%} of pixels")
    for ax in axes:
        ax.axis("off")
    handles = [plt.Rectangle((0, 0), 1, 1, color=COLOURS[c] / 255) for c in range(5)]
    fig.legend(handles, NAMES, loc="lower center", ncol=5)
    plt.show()


interact(seg_demo, scene=widgets.IntSlider(value=0, min=0, max=len(X_test) - 1), model=list(runs))

## 5. Instance segmentation

Semantic masks merge touching objects of the same class. **Instance** segmentation predicts one mask **per object**:

- **Mask R-CNN** (He et al., 2017) = Faster R-CNN (Week 8A) + a small FCN that predicts a 28 × 28 mask inside each RoI (RoIAlign keeps it aligned).
- **YOLO-seg** (YOLACT idea, Bolya et al., 2019): the network predicts a set of global **prototype masks** once per image, and every detection
  predicts **coefficients** that combine them: $M_i = \sigma\big(\sum_k c_{ik} P_k\big)$, cropped to its box. One pass, real time.

Instance masks are also how **labels for segmentation datasets** are stored: YOLO-seg uses one polygon per object (`class x1 y1 x2 y2 ...`, normalised).

In [ ]:
from ultralytics import YOLO

seg_model = YOLO(str(data_dir("weights") / "yolo11n-seg.pt"))
cap = cv2.VideoCapture(str(opencv_sample("vtest.avi"))); _, street = cap.read(); cap.release()
tests = {"messi5.jpg": load_sample("messi5.jpg"), "vtest.avi": street, "fruits.jpg": load_sample("fruits.jpg")}
fig, axes = plt.subplots(1, 3, figsize=(22, 5.5))
for ax, (name, im) in zip(axes, tests.items()):
    t0 = time.perf_counter()
    r = seg_model.predict(im, conf=0.3, verbose=False)[0]
    n = 0 if r.masks is None else len(r.masks)
    ax.imshow(r.plot()[..., ::-1]); ax.axis("off"); ax.set_title(f"{name}: {n} instance masks ({(time.perf_counter() - t0) * 1000:.0f} ms)")
plt.suptitle("Figure 5.1 - YOLO11n-seg (COCO): one mask per detected object", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# Mask R-CNN (torchvision, BSD licence): heavier but a strong baseline; downloads ~170 MB once
RUN_MASK_RCNN = not SMOKE
if RUN_MASK_RCNN:
    from torchvision.models.detection import MaskRCNN_ResNet50_FPN_V2_Weights, maskrcnn_resnet50_fpn_v2
    mrcnn, ok = load_pretrained(maskrcnn_resnet50_fpn_v2, MaskRCNN_ResNet50_FPN_V2_Weights.DEFAULT)
    mrcnn = mrcnn.eval().to(DEVICE)
    im = tests["messi5.jpg"]
    with torch.no_grad():
        out = mrcnn([torch.tensor(im[..., ::-1].copy()).permute(2, 0, 1).float().div(255).to(DEVICE)])[0]
    keep = out["scores"].cpu().numpy() > 0.5
    vis = im[..., ::-1].copy()
    for k, m in enumerate(out["masks"].cpu().numpy()[keep, 0]):
        vis[m > 0.5] = (0.5 * vis[m > 0.5] + 0.5 * np.array(plt.cm.tab10(k % 10)[:3]) * 255).astype(np.uint8)
    show(vis[..., ::-1], titles=[f"Mask R-CNN ({'pretrained' if ok else 'RANDOM weights - no internet'}): {keep.sum()} instances with score > 0.5"],
         figsize=(9, 6))

In [ ]:
def mask_to_polygon_ref(mask, img_w, img_h, cls=0, eps=1.5):
    cs, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    c = max(cs, key=cv2.contourArea)
    poly = cv2.approxPolyDP(c, eps, True).reshape(-1, 2).astype(float)
    return f"{cls} " + " ".join(f"{x / img_w:.5f} {y / img_h:.5f}" for x, y in poly)


img, b, l, masks = raw_test[1]
lines = [mask_to_polygon_ref(m, img.shape[1], img.shape[0], c) for m, c in zip(masks, l)]
vis = img.copy()
for ln in lines:
    v = np.array(ln.split()[1:], float).reshape(-1, 2) * [img.shape[1], img.shape[0]]
    cv2.polylines(vis, [v.astype(np.int32)], True, (0, 255, 255), 2)
show(vis, titles=[f"masks exported as {len(lines)} YOLO-seg polygon labels (yellow), e.g. '{lines[0][:60]}...'"], figsize=(10, 7))

## 6. Exercises (with self-checks)
Replace each `raise NotImplementedError`, then run the check below it.

### E1. Mean IoU
Return the **mean** IoU over the classes `0 .. n-1` for integer label maps `pred` and `gt`. A class absent from both maps should count as IoU 1 (nothing to get wrong).

In [ ]:
def mean_iou(pred, gt, n=5):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e1():
    gt = np.array([[0, 0, 1, 1], [0, 0, 1, 1]]); pred = np.array([[0, 0, 0, 1], [0, 0, 1, 1]])
    assert abs(mean_iou(pred, gt, 2) - (4 / 5 + 3 / 4) / 2) < 1e-9, "IoU(0) = 4/5, IoU(1) = 3/4"
    assert abs(mean_iou(gt, gt, 3) - 1.0) < 1e-9, "a perfect prediction has mIoU 1 (class 2 absent everywhere)"
    p = runs["U-Net, CE + Dice"]["pred"]
    assert abs(mean_iou(p, Y_test.numpy()) - per_class_iou(p, Y_test.numpy()).mean()) < 0.02, "differs from Section 4"


check("E1 mean_iou", _test_e1)

### E2. Soft Dice loss
Implement the multi-class soft Dice loss of Section 3 for `logits` `[B, C, H, W]` and integer `target` `[B, H, W]`: softmax, one-hot, per-class Dice summed over batch and pixels with smoothing `eps`, then `1 - mean over classes`. It must be differentiable.

In [ ]:
def soft_dice_loss(logits, target, n=5, eps=1.0):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e2():
    torch.manual_seed(0)
    logits = torch.randn(2, 5, 8, 8, requires_grad=True); target = torch.randint(0, 5, (2, 8, 8))
    loss = soft_dice_loss(logits, target)
    assert torch.is_tensor(loss) and loss.ndim == 0, "return a scalar tensor"
    assert abs(loss.item() - soft_dice_loss_ref(logits, target).item()) < 1e-5, "differs from the reference"
    loss.backward()
    assert logits.grad is not None and logits.grad.abs().sum() > 0, "the loss must be differentiable"
    perfect = F.one_hot(target, 5).permute(0, 3, 1, 2).float() * 50
    assert soft_dice_loss(perfect, target).item() < 0.01, "a perfect prediction should give ~0"


check("E2 soft_dice_loss", _test_e2)

### E3. Mask to YOLO-seg polygon
Return the label line `"cls x1 y1 x2 y2 ..."` for the **largest** external contour of a boolean `mask`, simplified with `cv2.approxPolyDP(contour, eps, True)` and normalised by the image size (5 decimals).

In [ ]:
def mask_to_polygon(mask, img_w, img_h, cls=0, eps=1.5):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e3():
    m = np.zeros((100, 200), bool); m[20:60, 50:150] = True; m[80:85, 5:10] = True
    ln = mask_to_polygon(m, 200, 100, cls=2)
    parts = ln.split()
    assert parts[0] == "2" and len(parts) == 1 + 2 * 4, f"a rectangle should give 4 vertices, got '{ln}'"
    xy = np.array(parts[1:], float).reshape(-1, 2)
    assert np.allclose(sorted(set(np.round(xy[:, 0], 3))), [0.25, 0.745], atol=0.01), "x coordinates must be normalised by the width"
    assert mask_to_polygon(masks[0], img.shape[1], img.shape[0], l[0]) == mask_to_polygon_ref(masks[0], img.shape[1], img.shape[0], l[0])


check("E3 mask_to_polygon", _test_e3)

### E4. Resolution and receptive field (no self-check)
Train the U-Net at half the resolution (80 × 104) and at the full 480 × 640 on a GPU (or 320 × 416 on a CPU). How do the hex-nut IoU and the
training time change? Then remove the deepest level (three scales instead of four). Relate the results to the receptive field and to the size of the
smallest object in pixels.

In [ ]:
# E4: your code here

## 7. Challenge: instance segmentation on the conveyor
1. Convert the conveyor scenes into a **YOLO-seg dataset** with `mask_to_polygon` (one `.txt` per image, like Week 9B) and fine-tune `yolo11n-seg.pt`.
2. Compare the mask IoU of touching parts with the U-Net + connected-components approach. Where does the semantic approach fail?
3. Measure the area of each washer in mm² from its mask (0.5 mm/px) and flag washers whose area deviates by more than 5%.

In [ ]:
# Your challenge code here

## 8. Quick check (auto-marked)

In [ ]:
quiz([
    {
        "q": 'Two touching bolts must be counted separately. Which task do you need?',
        "options": [
            'Semantic segmentation',
            'Instance segmentation',
            'Classification',
            'Edge detection',
        ],
        "answer": 'c1ea080c93',
        "explain": 'Semantic segmentation merges touching objects of the same class.',
    },
    {
        "q": "A model predicts 'belt' everywhere; belt is 96% of the pixels. Its pixel accuracy and mIoU (5 classes)?",
        "options": [
            '96% and 0.96',
            '96% and about 0.19',
            '4% and 0.04',
            '100% and 1.0',
        ],
        "answer": 'bda5268eb4',
        "explain": 'Belt IoU ~0.96, the four part classes 0: mean ~0.19.',
    },
    {
        "q": 'What do skip connections in a U-Net provide?',
        "options": [
            'More parameters',
            'High-resolution detail from the encoder to the decoder',
            'Faster training only',
            'Data augmentation',
        ],
        "answer": '4b70210ea3',
        "explain": 'Fine detail lost by pooling is restored.',
    },
    {
        "q": 'Why add a Dice loss to cross-entropy for segmentation?',
        "options": [
            'It is faster',
            'It weighs each class equally, whatever its pixel count',
            'It removes the need for labels',
            'It sharpens images',
        ],
        "answer": '6139322fba',
        "explain": 'Cross-entropy is dominated by the large background class.',
    },
    {
        "q": 'What does a dilated convolution do?',
        "options": [
            'Downsamples the image',
            'Enlarges the receptive field without extra parameters or downsampling',
            'Adds colour channels',
            'Normalises activations',
        ],
        "answer": 'dfa3b3d748',
        "explain": 'The kernel taps are spread apart.',
    },
    {
        "q": 'How does YOLO-seg produce a mask for each detection?',
        "options": [
            'A separate network per object',
            'Coefficients that combine shared prototype masks',
            'Thresholding the image',
            'Watershed',
        ],
        "answer": 'efbd584ab0',
        "explain": 'The YOLACT idea: prototypes once per image, coefficients per detection.',
    },
    {
        "q": 'If IoU = 0.6, what is the Dice coefficient?',
        "options": [
            '0.60',
            '0.75',
            '0.36',
            '0.43',
        ],
        "answer": '9f0a1cb414',
        "explain": 'Dice = 2 IoU / (1 + IoU) = 1.2 / 1.6 = 0.75.',
    },
], title='Week 11A quiz')

## 9. Exit ticket (reflection)
1. For your project, do you need semantic, instance or panoptic segmentation? Why?
2. Which class had the lowest IoU in Figure 4.1, and what two data or model changes would you try first?
3. When would classical segmentation (Week 4) still be the better choice?

## 10. What's new (2025–26)

> - **Universal segmentation**: Mask2Former and OneFormer handle semantic, instance and panoptic segmentation with one architecture (mask
>   classification with transformer queries, like DETR).
> - **Foundation segmenters**: SAM, SAM 2 and SAM 3 (promptable by points, boxes or text concepts), and open-vocabulary segmenters (Grounded-SAM,
>   YOLOE-seg) produce masks for objects they were never trained on — Session B.
> - **Efficient real-time segmentation** (YOLO11-seg, RT-DETR-seg variants, SegFormer, EfficientViT-SAM) runs on edge GPUs and NPUs.
> - **Label-efficient training**: SAM-generated pseudo-labels plus a small human-checked set is now the usual way to build segmentation datasets.

## 11. References and further exploration
- Long, J., Shelhamer, E. & Darrell, T. (2015). *Fully convolutional networks for semantic segmentation.* CVPR. [arXiv:1411.4038](https://arxiv.org/abs/1411.4038)
- Ronneberger, O., Fischer, P. & Brox, T. (2015). *U-Net: Convolutional networks for biomedical image segmentation.* MICCAI. [arXiv:1505.04597](https://arxiv.org/abs/1505.04597)
- Chen, L.-C. et al. (2018). *Encoder-decoder with atrous separable convolution* (DeepLabv3+). ECCV. [arXiv:1802.02611](https://arxiv.org/abs/1802.02611)
- Yu, F. & Koltun, V. (2016). *Multi-scale context aggregation by dilated convolutions.* ICLR. [arXiv:1511.07122](https://arxiv.org/abs/1511.07122)
- Milletari, F., Navab, N. & Ahmadi, S.-A. (2016). *V-Net* (soft Dice loss). 3DV. [arXiv:1606.04797](https://arxiv.org/abs/1606.04797)
- He, K. et al. (2017). *Mask R-CNN.* ICCV. [arXiv:1703.06870](https://arxiv.org/abs/1703.06870)
- Bolya, D. et al. (2019). *YOLACT: Real-time instance segmentation.* ICCV. [arXiv:1904.02689](https://arxiv.org/abs/1904.02689)
- Kirillov, A. et al. (2019). *Panoptic segmentation.* CVPR. [arXiv:1801.00868](https://arxiv.org/abs/1801.00868)
- Cheng, B. et al. (2022). *Masked-attention mask transformer for universal image segmentation* (Mask2Former). CVPR. [arXiv:2112.01527](https://arxiv.org/abs/2112.01527)
- Xie, E. et al. (2021). *SegFormer.* NeurIPS. [arXiv:2105.15203](https://arxiv.org/abs/2105.15203)
- Prince (2023), *Understanding Deep Learning*, §10–11. [udlbook.github.io](https://udlbook.github.io/udlbook/) · Ultralytics *Segment* docs. [docs.ultralytics.com/tasks/segment](https://docs.ultralytics.com/tasks/segment/)

**Data:** conveyor scenes from `partsim.py`; `messi5.jpg`, `vtest.avi`, `fruits.jpg` from the OpenCV sample data; YOLO11n-seg (AGPL-3.0) and Mask R-CNN (BSD-3) COCO weights.

## 12. Key takeaways
- Semantic = class per pixel; instance = mask per object; panoptic = both.
- Encoder–decoders recover resolution with **up-sampling**, **skip connections** (U-Net) and **dilated** convolutions.
- Evaluate with **per-class IoU / mIoU / Dice**; train imbalanced problems with **CE + Dice**.
- **Mask R-CNN** (mask per RoI) and **YOLO-seg** (prototypes × coefficients) give instance masks; polygons store them as labels.

**Next session (11B):** segment *anything* with SAM, and detect defects never seen in training with **anomaly detection**.